In [10]:
import numpy as np
import numpy.testing as npt
from scipy import stats
from scipy.stats import t,ttest_ind
from scipy.stats import f
from scipy.stats import f_oneway
import statsmodels.api as sm
from statsmodels.regression._prediction import get_prediction
from statsmodels.stats.outliers_influence import OLSInfluence,MLEInfluence
from statsmodels.graphics.gofplots import qqplot_2samples,ProbPlot,qqplot
import pandas as pd
from patsy import dmatrices
from numpy.testing import assert_almost_equal, assert_allclose
import matplotlib.pyplot as plt
from mpl_toolkits.axes_grid1.inset_locator import inset_axes
import seaborn as sns

# some_file.py
import sys
# caution: path[0] is reserved for script path (or '' in REPL)
sys.path.insert(1, r'C:\Users\TODO\Desktop\Abhi\AI\AI\Math\Hands-On\statemodelsStudy')
from olsRegressionAnalysis import dispAnalysisOfVariance, tableDispFormatt,getInvOfProductMat,getRegressionEqn,\
                                  dispReghressionAnalysis,getCorrelation

In [11]:
path  = r"C:\Users\TODO\Desktop\Abhi\AI\AI\Math\Hands-On\Notes\mlx\stat_analysis\CH-8-INDICATOR_VARIABLES\IndecatorVarPythonCode\Dataset\ToolLifeDataTbl_8_1.txt"
df = pd.read_csv(path)
print(df.columns)
ser = pd.Series({'A':0,'B':1})
df['Encode'] = df['ToolType'].map(ser)
df['CrossProd'] = df['Encode'].multiply(df['x_rpm'])
print(df.head())
dfX = df.loc[:,['y_hours', 'x_rpm']]

Index(['y_hours', 'x_rpm', 'ToolType'], dtype='object')
   y_hours  x_rpm ToolType  Encode  CrossProd
0    18.73    610        A       0          0
1    14.52    950        A       0          0
2    17.43    720        A       0          0
3    14.54    840        A       0          0
4    13.44    980        A       0          0


In [12]:
# Check If Multicolinary exist


ret = getCorrelation(data_frame = dfX ,CorrelationsThreshold = 0.9)
if ret.empty == False:
    plt.show()

y, X = dmatrices(
                 formula_like = ' y_hours ~ x_rpm + Encode + CrossProd', 
                 data=df,
                 return_type='dataframe'
                 )
res = sm.OLS(y, X).fit()

=============================== Correlation between groups =================================
            between      Corr    pVal
0  [y_hours, x_rpm] -0.431306  0.0576
=============================== Check if High Correlation Exist between groups =============
None of the pairwise correlations Rij are suspiciously large


In [13]:
res = sm.OLS(y, X).fit()
SSres_fm = res.ssr
MSres_fm = res.mse_model
SSr_fm   = res.ess
MSr_fm   = res.mse_resid
df_full_models = res.df_model # number of group
dfd_fm = res.df_resid
tableDispFormatt('Analysis of Variance')
dispAnalysisOfVariance(resResult = res)
dispReghressionAnalysis(res)
getRegressionEqn(res)

=============================== Analysis of Variance =======================================
=============================== Analysis of Variance =======================================
  Source Of Var  DegOfFreedom(DF)  Sum Of Square(SS)  Mean Square(MS)          F      FSig    P
0    Regression               3.0        1434.112351       478.037450  54.254685  3.238872  0.0
1      Residual              16.0         140.975829         8.810989        ---       ---  ---
2         Total              19.0        1575.088180       486.848440        ---       ---  ---
r-square:  0.9104965480064204 rSqr-adj 0.8937146507576242 rSqr-Predict
=============================== Regression Equation ========================================
=============================== Regression eqn =============================================
32.775  -0.021 x_rpm + 23.971 Encode -0.012 CrossProd 
=============================== Regression Analysis ========================================
             RegCoff   SE

'32.775  -0.021 x_rpm + 23.971 Encode -0.012 CrossProd '

Before going any test Understaqnd the models:

ToolType = A & B

           assume A = 0 & B = 1

Let suppose say

    x1 =  x_rpm,  x2 = toolType  CrossProd = x1*x2 correcponding 

    y = y_hours

    regressor cofficinet b1, b2, b3 (b stand for beta)

Compleate models eqn

   y = b0 + b1x1 + b2x2 + b3x1x3 + ϵ ----(i)

Now eqn (i) Due to tool type A, put x1 = 0 in (i)

   y = b0 + b2x2 + ϵ ----(ii)    line one eqn

    Intercept = b0 & Slope = b2

Now eqn (i) Due to tool type B, put x1 = 1 in (i)

   y = (b0 + b2) + (b1+b3)x1 + ϵ ----(iii)    line two eqn  

Test 1:

test the hypothesis that the two regression lines are identical:

Null Hypo = beta(crossprod) , beta(Encode) = 0 i.e. Both line are identicals
Alternate hypo = beta(crossprod) , beta(Encode) ≠ 0 i.e. Both line are not identicals

In [14]:
y, X = dmatrices(
                 formula_like = ' y_hours ~ x_rpm', 
                 data=df,
                 return_type='dataframe'
                 )
res = sm.OLS(y, X).fit()
SSres_rm = res.ssr
MSres_rm = res.mse_model
SSr_rm   = res.ess
MSr_rm   = res.mse_resid

In [15]:
# Calculate SSr due to beta(crossprod) & beta(Encode)
SSReffect = SSr_fm - SSr_rm 
MSresDueToFullModels = MSr_fm
# Step 4: Calculate F score
r = 2 # Calculating effect of two variable in rediused models
F = (SSReffect/r)/MSresDueToFullModels
# Calculate significance 95%
FSig = f.isf(q = 0.05, dfn = r,dfd = dfd_fm, loc=0, scale=1)
f_rm_pvalue = f.sf(F, dfn = r,dfd = dfd_fm)
tableDispFormatt('regression lines are identical')
print('Fstae: ',F, ' FSig: ',FSig, ' p-val: ',f_rm_pvalue)

=============================== regression lines are identical =============================
Fstae:  64.75475706952773  FSig:  3.63372346759163  p-val:  2.137119152971871e-08



Result Analysis:

if F > FSig or f_rm_pvalue ≤ 0.05 reject null hypothesis

       i.e. both lines are not identicals



Test 2:

test the hypothesis that the two regression lines are identical in slope:

Null Hypo = b3 = 0 i.e. Both line are identicals in slope (see both line eqn)

Alternate hypo = b3 ≠ 0 i.e. Both line have not identicals slope

SSr(b3|b2,b1,b0) = SSr(b0,b1,b2,b3) - SSr(b0,b1,b2)

In [16]:
# Get redused models parameters
y, X = dmatrices(
                 formula_like = ' y_hours ~ x_rpm + Encode', 
                 data=df,
                 return_type='dataframe'
                 )
res = sm.OLS(y, X).fit()
SSres_rm = res.ssr
MSres_rm = res.mse_model
SSr_rm   = res.ess
MSr_rm   = res.mse_resid

In [17]:
# Calculate SSr due to beta(crossprod) & beta(Encode)
SSReffect = SSr_fm - SSr_rm 
MSresDueToFullModels = MSr_fm
# Step 4: Calculate F score
r = 1 # Calculating effect of two variable in rediused models
F = (SSReffect/r)/MSresDueToFullModels
# Calculate significance 95%
FSig = f.isf(q = 0.05, dfn = r,dfd = dfd_fm, loc=0, scale=1)
f_rm_pvalue = f.sf(F, dfn = r,dfd = dfd_fm)
tableDispFormatt('regression lines are identical in slope')
print('Fstae: ',F, ' FSig: ',FSig, ' p-val: ',f_rm_pvalue)

=============================== regression lines are identical in slope ====================
Fstae:  1.8248499058350929  FSig:  4.493998477666352  p-val:  0.19553298300424074



Result Analysis:

Fstate < FSig && f_rm_pvalue > 0.05

Faile to reject Null hypthesis. Can say slopes of the

two straight lines are the same.

Another way to say: ???

Test 3: TODO

test the hypothesis that the two regression lines are identical in slope:

Null Hypo = b3 = 0 i.e. Both line are identicals in slope (see both line eqn)
Alternate hypo = b3 ≠ 0 i.e. Both line have not identicals slope

SSr(b3|b2,b1,b0) = SSr(b0,b1,b2,b3) - SSr(b0,b1,b2)


TODO BOOK TABLE 8.3 PG 490 PDF versions
